In [ ]:
import sys
import subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--no-cache-dir",
    "--force-reinstall",
    "torch==2.6.0",
    "torchvision==0.21.0",
    "torchaudio==2.6.0",
    "--index-url", "https://download.pytorch.org/whl/cu124"
], check=True)

print("Đã cài PyTorch tương thích P100. Hãy Restart Session.")

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Architectures:", torch.cuda.get_arch_list())

assert torch.cuda.is_available()
assert "sm_60" in torch.cuda.get_arch_list()

x = torch.randn(1024, 1024, device="cuda")
y = x @ x
print("P100 CUDA test thành công:", y.device)

In [ ]:
# ========================= USER CONFIG =========================
# Train UCF
DATASET = "ucf"
MAX_EPOCHS = 10           
UCF_BATCH_SIZE = 32    
NUM_WORKERS = 2
SEED = 234
RESUME = False
WARM_START = False 
# ĐỔI LINK REPO SANG BẢN FIX
REPO_URL = "https://github.com/Sharvuz/RUPA-DSA-fix.git"
REPO_REF = None

In [ ]:
import json, os, random, shutil, subprocess, sys, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "RUPA-DSA"
ARTIFACTS = WORK / "rupa_artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    shutil.rmtree(REPO)
clone_cmd = ["git", "clone", "--depth", "1"]
if REPO_REF:
    clone_cmd += ["--branch", REPO_REF]
clone_cmd += [REPO_URL, str(REPO)]
subprocess.run(clone_cmd, check=True)
os.chdir(REPO)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ftfy", "regex", "einops==0.8.0", "ipdb", "scikit-learn", "pandas"
], check=True)

import numpy as np
import pandas as pd
import torch

print("Repository:", REPO_URL, "ref:", REPO_REF or "default branch")
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Enable Accelerator = GPU in Kaggle Notebook Settings."
assert "RUPA-DSA" in (REPO / "src/model.py").read_text(encoding="utf-8")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("Direct GitHub clone ready.")

In [ ]:
from collections import defaultdict
from pathlib import PurePosixPath

if DATASET == "ucf":
    csv_specs = [
        ("list/ucf_CLIP_rgb.csv", "ucf_train.csv"),
        ("list/ucf_CLIP_rgbtest.csv", "ucf_test.csv"),
    ]
    zip_hints = ["ucfclipfeatures"]
else:
    csv_specs = [
        ("list/xd_CLIP_rgb.csv", "xd_train.csv"),
        ("list/xd_CLIP_rgbtest.csv", "xd_test.csv"),
    ]
    zip_hints = ["xdtrainclipfeatures", "xdtestclipfeatures"]

def basename(path_string):
    return PurePosixPath(str(path_string).replace("\\", "/")).name

required_names = set()
for csv_rel, _ in csv_specs:
    frame = pd.read_csv(REPO / csv_rel)
    required_names.update(basename(p) for p in frame["path"])

feature_roots = [INPUT]

def indexed_names(roots):
    return {p.name for root in roots for p in root.rglob("*.npy")}

missing_before = required_names - indexed_names(feature_roots)
if missing_before:
    extract_root = WORK / f"rupa_features_{DATASET}"
    extract_root.mkdir(parents=True, exist_ok=True)
    archives = [
        p for p in INPUT.rglob("*.zip")
        if any(hint in p.stem.lower() for hint in zip_hints)
    ]
    if archives:
        print(f"Extracting {len(archives)} feature archive(s); this can take several minutes...")
    for archive_path in archives:
        destination = (extract_root / archive_path.stem).resolve()
        destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                target = (destination / member.filename).resolve()
                if destination not in target.parents and target != destination:
                    raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
            archive.extractall(destination)
        print("Extracted:", archive_path.name)
    feature_roots.append(extract_root)

index = defaultdict(list)
for root in feature_roots:
    for path in root.rglob("*.npy"):
        if path.name in required_names:
            index[path.name].append(path)

def rank_candidate(candidate, original):
    original_parts = [x.lower() for x in str(original).replace("\\", "/").split("/")[-4:]]
    candidate_text = str(candidate).lower()
    return sum(part in candidate_text for part in original_parts)

def rewrite_csv(csv_rel, output_name):
    frame = pd.read_csv(REPO / csv_rel)
    resolved, missing, ambiguous = [], [], 0
    for original in frame["path"].astype(str):
        matches = index.get(basename(original), [])
        if not matches:
            missing.append(basename(original))
            resolved.append("")
            continue
        matches = sorted(matches, key=lambda p: rank_candidate(p, original), reverse=True)
        resolved.append(str(matches[0]))
        ambiguous += int(len(matches) > 1)
    if missing:
        raise FileNotFoundError(
            f"{csv_rel}: missing {len(missing)}/{len(frame)} features; examples={missing[:10]}"
        )
    output = ARTIFACTS / output_name
    frame["path"] = resolved
    frame.to_csv(output, index=False)
    print(f"{output_name}: rows={len(frame)}, missing=0, duplicate basenames={ambiguous}")
    return output

train_csv = rewrite_csv(*csv_specs[0])
test_csv = rewrite_csv(*csv_specs[1])

sample_path = Path(pd.read_csv(train_csv).iloc[0]["path"])
sample = np.load(sample_path, mmap_mode="r")
assert sample.ndim == 2 and sample.shape[-1] == 512, (
    f"Expected CLIP feature [T,512], got {sample.shape} from {sample_path}"
)
print("Feature preflight:", sample_path.name, sample.shape)

In [ ]:
# Train UCF
import json, os, subprocess, sys, time
from pathlib import Path

dataset_dir = ARTIFACTS / DATASET
dataset_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = dataset_dir / f"checkpoint_{DATASET}.pth"
model_path = dataset_dir / f"best_{DATASET}.pth"
train_csv = ARTIFACTS / f"{DATASET}_train.csv"
test_csv = ARTIFACTS / f"{DATASET}_test.csv"

cmd = [
    sys.executable, f"src/{DATASET}_train.py",
    "--train-list", str(train_csv),
    "--test-list", str(test_csv),
    "--model-path", str(model_path),
    "--checkpoint-path", str(checkpoint_path),
    
    "--max-epoch", str(MAX_EPOCHS),
    "--batch-size", str(UCF_BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--seed", str(SEED),
    
    # Bật tính năng RUPA và Otsu
    "--rupa-use", "true",

    # Các trọng số mặc định chuẩn xác của bản v2
    "--routing-det-weight", "0.5",
    "--routing-rec-weight", "0.3",
    "--routing-sem-weight", "0.2",
    "--loss-residual-weight", "1.0",
    "--loss-reconstructed-normal-weight", "1.0",
    "--loss-dnp-normal-weight", "0.1",
    "--loss-consistency-weight", "1.0",
    "--loss-gather-weight", "1.0",
    
    # THÔNG SỐ TỐI ƯU CHO BATCH SIZE 32 (Linear Scaling & EMA tuning)
    "--lr", "1.4e-4",
    "--ema_momentum", "0.5",
    "--scheduler-milestones", "3", "5",
    "--scheduler-gamma", "0.1",
]

if RESUME:
    cmd.extend(["--use-checkpoint", "true"])

log_path = dataset_dir / f"train_{DATASET}_fix_bs32.log"
started = time.time()
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

print(f"Bắt đầu train UCF-Crime bản Fix với {MAX_EPOCHS} Epochs (Cấu hình BS 32)...")

with log_path.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        cmd, cwd=REPO, env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f"Training failed with exit code {return_code}")

print(f"Hoàn thành xuất sắc trong {(time.time() - started)/3600:.2f} giờ!")

# XD-violence


In [ ]:
!python src/xd_train.py \
    --train-list /kaggle/working/rupa_artifacts/xd_train.csv \
    --test-list /kaggle/working/rupa_artifacts/xd_test.csv \
    --model-path /kaggle/working/rupa_artifacts/xd/best_xd.pth \
    --checkpoint-path /kaggle/working/rupa_artifacts/xd/checkpoint_xd.pth \
    --max-epoch 10 \
    --batch-size 64 \
    --num-workers 2 \
    --seed 234 \
    --rupa-use true \
    --adaptive_normal_selection true \
    --routing-det-weight 0.5 \
    --routing-rec-weight 0.3 \
    --routing-sem-weight 0.2 \
    --loss-residual-weight 1.0 \
    --loss-reconstructed-normal-weight 1.0 \
    --loss-dnp-normal-weight 0.1 \
    --loss-consistency-weight 1.0 \
    --loss-gather-weight 1.0 \
    --lr 2e-5 \
    --ema_momentum 0.9 \
    --scheduler-milestones 5 8 \
    --scheduler-gamma 0.1

# p====================================================

# final

In [ ]:
# ========================= USER CONFIG =========================
DATASET = "ucf"             
MAX_EPOCHS = 5             
UCF_BATCH_SIZE = 48      # Nếu bị tràn RAM, bạn hãy hạ xuống 32 nhé
NUM_WORKERS = 2
SEED = 234
RESUME = False
# Khai báo đường dẫn tới GitHub (Sửa lỗi NameError)
REPO_URL = "https://github.com/Sharvuz/RUPA-DSA_final.git"
REPO_REF = None
# [QUAN TRỌNG] ĐƯỜNG DẪN ĐẾN FILE CHECKPOINT DSANET GỐC DÙNG LÀM ĐÀ PHÓNG
INIT_MODEL_PATH = "/kaggle/input/models/huynhvu7979/dsanet/pytorch/default/1/dsanet_model_ucf.pth"

In [ ]:
import json, os, random, shutil, subprocess, sys, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "RUPA-DSA"
ARTIFACTS = WORK / "rupa_artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    shutil.rmtree(REPO)
clone_cmd = ["git", "clone", "--depth", "1"]
if REPO_REF:
    clone_cmd += ["--branch", REPO_REF]
clone_cmd += [REPO_URL, str(REPO)]
subprocess.run(clone_cmd, check=True)
os.chdir(REPO)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ftfy", "regex", "einops==0.8.0", "ipdb", "scikit-learn", "pandas"
], check=True)

import numpy as np
import pandas as pd
import torch

print("Repository:", REPO_URL, "ref:", REPO_REF or "default branch")
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Enable Accelerator = GPU in Kaggle Notebook Settings."
assert "RUPA-DSA" in (REPO / "src/model.py").read_text(encoding="utf-8")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("Direct GitHub clone ready.")

In [ ]:
from collections import defaultdict
from pathlib import PurePosixPath

if DATASET == "ucf":
    csv_specs = [
        ("list/ucf_CLIP_rgb.csv", "ucf_train.csv"),
        ("list/ucf_CLIP_rgbtest.csv", "ucf_test.csv"),
    ]
    zip_hints = ["ucfclipfeatures"]
else:
    csv_specs = [
        ("list/xd_CLIP_rgb.csv", "xd_train.csv"),
        ("list/xd_CLIP_rgbtest.csv", "xd_test.csv"),
    ]
    zip_hints = ["xdtrainclipfeatures", "xdtestclipfeatures"]

def basename(path_string):
    return PurePosixPath(str(path_string).replace("\\", "/")).name

required_names = set()
for csv_rel, _ in csv_specs:
    frame = pd.read_csv(REPO / csv_rel)
    required_names.update(basename(p) for p in frame["path"])

feature_roots = [INPUT]

def indexed_names(roots):
    return {p.name for root in roots for p in root.rglob("*.npy")}

missing_before = required_names - indexed_names(feature_roots)
if missing_before:
    extract_root = WORK / f"rupa_features_{DATASET}"
    extract_root.mkdir(parents=True, exist_ok=True)
    archives = [
        p for p in INPUT.rglob("*.zip")
        if any(hint in p.stem.lower() for hint in zip_hints)
    ]
    if archives:
        print(f"Extracting {len(archives)} feature archive(s); this can take several minutes...")
    for archive_path in archives:
        destination = (extract_root / archive_path.stem).resolve()
        destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                target = (destination / member.filename).resolve()
                if destination not in target.parents and target != destination:
                    raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
            archive.extractall(destination)
        print("Extracted:", archive_path.name)
    feature_roots.append(extract_root)

index = defaultdict(list)
for root in feature_roots:
    for path in root.rglob("*.npy"):
        if path.name in required_names:
            index[path.name].append(path)

def rank_candidate(candidate, original):
    original_parts = [x.lower() for x in str(original).replace("\\", "/").split("/")[-4:]]
    candidate_text = str(candidate).lower()
    return sum(part in candidate_text for part in original_parts)

def rewrite_csv(csv_rel, output_name):
    frame = pd.read_csv(REPO / csv_rel)
    resolved, missing, ambiguous = [], [], 0
    for original in frame["path"].astype(str):
        matches = index.get(basename(original), [])
        if not matches:
            missing.append(basename(original))
            resolved.append("")
            continue
        matches = sorted(matches, key=lambda p: rank_candidate(p, original), reverse=True)
        resolved.append(str(matches[0]))
        ambiguous += int(len(matches) > 1)
    if missing:
        raise FileNotFoundError(
            f"{csv_rel}: missing {len(missing)}/{len(frame)} features; examples={missing[:10]}"
        )
    output = ARTIFACTS / output_name
    frame["path"] = resolved
    frame.to_csv(output, index=False)
    print(f"{output_name}: rows={len(frame)}, missing=0, duplicate basenames={ambiguous}")
    return output

train_csv = rewrite_csv(*csv_specs[0])
test_csv = rewrite_csv(*csv_specs[1])

sample_path = Path(pd.read_csv(train_csv).iloc[0]["path"])
sample = np.load(sample_path, mmap_mode="r")
assert sample.ndim == 2 and sample.shape[-1] == 512, (
    f"Expected CLIP feature [T,512], got {sample.shape} from {sample_path}"
)
print("Feature preflight:", sample_path.name, sample.shape)

In [ ]:
# ========================= TEST MODEL =========================
import subprocess, sys, os
from pathlib import Path

# Đã sửa lại đúng đường dẫn Kaggle Dataset mới của bạn
TEST_MODEL_PATH = "/kaggle/input/models/huynhvu7979/dsanet/pytorch/default/1/dsanet_model_ucf.pth"

# Khai báo lại đường dẫn phòng hờ
DATASET = "ucf"
REPO_DIR = "/kaggle/working/RUPA-DSA"
ARTIFACTS = Path("/kaggle/working/rupa_artifacts")
test_csv = ARTIFACTS / f"{DATASET}_test.csv"

# 1. Tự động Clone code nếu chưa có
if not os.path.exists(REPO_DIR):
    print("📥 Đang tải mã nguồn từ GitHub về Kaggle...", flush=True)
    subprocess.run(["git", "clone", "https://github.com/Sharvuz/RUPA-DSA_final.git", REPO_DIR])

cmd_test = [
    sys.executable, "-u", "src/ucf_test.py",
    "--test-list", str(test_csv),
    "--model-path", str(TEST_MODEL_PATH),
    
    # RẤT QUAN TRỌNG: 
    # Vì file này là DSANet gốc, KHÔNG có cấu trúc RUPA bên trong
    # Nên bắt buộc phải tắt RUPA để chấm điểm chính xác sức mạnh gốc của nó!
    "--rupa-use", "false", 
    "--DNP_use", "true"
]

print(f"\n🔍 Bắt đầu kiểm tra sức mạnh của model: {TEST_MODEL_PATH} ...\n", flush=True)

process_test = subprocess.Popen(
    cmd_test, cwd=REPO_DIR, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
)

for line in process_test.stdout:
    print(line, end="")
    sys.stdout.flush()

process_test.wait()

## 🔍 Bắt đầu kiểm tra sức mạnh của model: /kaggle/input/models/huynhvu7979/dsanet/pytorch/default/1/dsanet_model_ucf.pth ...

### AUC1:  0.8953996818892952  AP1:  0.37849203810250753
### AUC2:  0.8953996999604814  AP2: 0.378486630436927
### mAP@0.1 =20.54%
### mAP@0.2 =13.12%
### mAP@0.3 =9.88%
### mAP@0.4 =8.57%
### mAP@0.5 =6.72%
### average MAP: 11.76

In [ ]:
# ========================= TRAINING =========================
import json, os, subprocess, sys, time
from pathlib import Path
# Định nghĩa đường dẫn
ARTIFACTS = Path("/kaggle/working/rupa_artifacts")
REPO = Path("/kaggle/working/RUPA-DSA")
dataset_dir = ARTIFACTS / DATASET
dataset_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = dataset_dir / f"checkpoint_{DATASET}.pth"
model_path = dataset_dir / f"best_{DATASET}.pth"
train_csv = ARTIFACTS / f"{DATASET}_train.csv"
test_csv = ARTIFACTS / f"{DATASET}_test.csv"
cmd = [
    sys.executable, "-u", f"src/{DATASET}_train.py",
    "--train-list", str(train_csv),
    "--test-list", str(test_csv),
    "--model-path", str(model_path),
    "--checkpoint-path", str(checkpoint_path),
    
    "--max-epoch", str(MAX_EPOCHS),
    "--batch-size", str(UCF_BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--seed", str(SEED),
    # ------------------ BẬT RUPA FINAL ------------------
    "--rupa-use", "true",
    
    # 1. Bật Otsu Thresholding (Dùng thuật toán cắt ngưỡng Otsu)
    "--adaptive_normal_selection", "true",
    
    # 2. Bật Warm Start (Kế thừa tri thức từ file DSANet gốc)
    "--init-model-path", str(INIT_MODEL_PATH),
    
    # 3. Bật Safe Gate cho Routing
    "--routing-mode", "safe_gate",
    "--main-lr", "0.0",         # Đóng băng nhánh DSANet
    "--refiner-lr", "1e-5",     # Chỉ cho phép nhánh RUPA học tiếp
    # Các trọng số RUPA
    "--routing-det-weight", "0.5",  
    "--routing-rec-weight", "0.3",   
    "--routing-sem-weight", "0.2",   
    "--loss-residual-weight", "1.0",
    "--loss-reconstructed-normal-weight", "1.0",
    "--loss-dnp-normal-weight", "0.1",
    "--loss-consistency-weight", "1.0",
    "--loss-gather-weight", "1.0",
]
if RESUME:
    cmd.extend(["--use-checkpoint", "true"])
log_path = dataset_dir / f"train_{DATASET}_FINAL.log"
started = time.time()
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
print(f"🚀 Bắt đầu train bản FINAL (Otsu + WarmStart) với {MAX_EPOCHS} Epochs...\n", flush=True)
with log_path.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        cmd, cwd=REPO, env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        sys.stdout.flush()
        log_file.write(line)
        log_file.flush()
        
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"❌ Training thất bại với exit code {return_code}")
print(f"\n✅ Hoàn thành xuất sắc trong {(time.time() - started)/3600:.2f} giờ!")

In [1]:
# ========================= USER CONFIG CHO XD-VIOLENCE =========================
DATASET = "xd"             
MAX_EPOCHS = 10            # XD-Violence phức tạp hơn nên để 10 Epochs
XD_BATCH_SIZE = 64         # Máy bạn gánh được 64 thì cứ mạnh dạn dùng 64
NUM_WORKERS = 2
SEED = 234
RESUME = False

REPO_URL = "https://github.com/Sharvuz/RUPA-DSA_final.git"
REPO_REF = None

# [RẤT QUAN TRỌNG] ĐƯỜNG DẪN TỚI FILE DSANET GỐC CỦA TẬP XD-VIOLENCE
INIT_MODEL_PATH = "/kaggle/input/models/huynhvu7979/dsanet/pytorch/default/1/dsanet_model_xd.pth"

In [2]:
import json, os, random, shutil, subprocess, sys, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "RUPA-DSA"
ARTIFACTS = WORK / "rupa_artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    shutil.rmtree(REPO)
clone_cmd = ["git", "clone", "--depth", "1"]
if REPO_REF:
    clone_cmd += ["--branch", REPO_REF]
clone_cmd += [REPO_URL, str(REPO)]
subprocess.run(clone_cmd, check=True)
os.chdir(REPO)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ftfy", "regex", "einops==0.8.0", "ipdb", "scikit-learn", "pandas"
], check=True)

import numpy as np
import pandas as pd
import torch

print("Repository:", REPO_URL, "ref:", REPO_REF or "default branch")
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Enable Accelerator = GPU in Kaggle Notebook Settings."
assert "RUPA-DSA" in (REPO / "src/model.py").read_text(encoding="utf-8")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("Direct GitHub clone ready.")

Cloning into '/kaggle/working/RUPA-DSA'...


Repository: https://github.com/Sharvuz/RUPA-DSA_final.git ref: default branch
PyTorch: 2.6.0+cu124
CUDA runtime: 12.4
GPU: Tesla P100-PCIE-16GB
Direct GitHub clone ready.


In [3]:
import os
import shutil
from pathlib import Path

print("Đang quét toàn bộ hệ thống Kaggle để tìm file .npy...")

# Quét sạch sẽ toàn bộ /kaggle/input và /kaggle/working
all_npy_paths = list(Path("/kaggle/input").rglob("*.npy")) + list(Path("/kaggle/working").rglob("*.npy"))

if not all_npy_paths:
    print("CẢNH BÁO ĐỎ: KHÔNG TÌM THẤY FILE .NPY NÀO! Bạn đã Add Data XD-Violence vào Kaggle chưa?")
else:
    print(f"Radar đã quét thấy {len(all_npy_paths)} file .npy trên hệ thống!")

# Tạo từ điển bản đồ: Tên file -> Đường dẫn tuyệt đối
file_map = {path.name: path for path in all_npy_paths}

def rewrite_csv(csv_rel, output_name):
    input_csv = REPO / csv_rel
    output = ARTIFACTS / output_name
    print(f"Đang đồng bộ danh sách {input_csv.name}...")
    
    missing = []
    is_first_line = True
    
    with input_csv.open("r", encoding="utf-8") as f, output.open("w", encoding="utf-8") as out:
        for line in f:
            line = line.strip()
            if not line: continue
            
            parts = line.split(",")
            feature_path = parts[0]
            
            if is_first_line and ("path" in feature_path.lower() or "name" in feature_path.lower()):
                out.write(line + "\n")
                is_first_line = False
                continue
            is_first_line = False
            
            feature_basename = Path(feature_path).name
            
            # Thử mọi cách "biến dạng" do Kaggle tự ý đổi tên file
            possible_names = [
                feature_basename,                                # Tên gốc chuẩn
                feature_basename.replace("#", "+"),              # Kaggle đổi # thành +
                feature_basename.replace("#", ""),               # Kaggle ăn mất dấu #
                feature_basename.replace("#", "_"),              # Kaggle đổi # thành _
            ]
            
            match_found = False
            for p_name in possible_names:
                if p_name in file_map:
                    parts[0] = str(file_map[p_name])
                    out.write(",".join(parts) + "\n")
                    match_found = True
                    break
                    
            if not match_found:
                missing.append(feature_basename)
                
    if missing:
        raise FileNotFoundError(f"Lỗi: Vẫn thiếu {len(missing)} file. Ví dụ: {missing[:3]}")
        
    return output

# Tiến hành khớp nối
train_csv = rewrite_csv(f"list/{DATASET}_CLIP_rgb.csv", f"{DATASET}_train.csv")
test_csv = rewrite_csv(f"list/{DATASET}_CLIP_rgbtest.csv", f"{DATASET}_test.csv")

print("✅ Đã lập bản đồ và đồng bộ dữ liệu XD-Violence thành công 100%!")

Đang quét toàn bộ hệ thống Kaggle để tìm file .npy...
Radar đã quét thấy 67046 file .npy trên hệ thống!
Đang đồng bộ danh sách xd_CLIP_rgb.csv...
Đang đồng bộ danh sách xd_CLIP_rgbtest.csv...
✅ Đã lập bản đồ và đồng bộ dữ liệu XD-Violence thành công 100%!


In [ ]:
# ========================= TEST MODEL XD-VIOLENCE =========================
import subprocess, sys, os
from pathlib import Path

# [QUAN TRỌNG] Đảm bảo bạn điền đúng đường dẫn file DSANet gốc của tập XD
TEST_MODEL_PATH = "/kaggle/input/models/huynhvu7979/dsanet/pytorch/default/1/dsanet_model_xd.pth"

# Khai báo đường dẫn
DATASET = "xd"
REPO_DIR = "/kaggle/working/RUPA-DSA"
ARTIFACTS = Path("/kaggle/working/rupa_artifacts")
test_csv = ARTIFACTS / f"{DATASET}_test.csv"

# Tự động Clone code nếu Kaggle vừa bị Restart
if not os.path.exists(REPO_DIR):
    print("📥 Đang tải mã nguồn từ GitHub về Kaggle...", flush=True)
    subprocess.run(["git", "clone", "https://github.com/Sharvuz/RUPA-DSA_final.git", REPO_DIR])

cmd_test = [
    sys.executable, "-u", "src/xd_test.py",
    "--test-list", str(test_csv),
    "--model-path", str(TEST_MODEL_PATH),
    
    # RẤT QUAN TRỌNG: 
    # - Nếu test file DSANet gốc (như hiện tại) -> Để "false"
    # - Nếu test file best_xd.pth của bạn (sau khi train xong) -> Nhớ sửa thành "true"
    "--rupa-use", "false", 
    "--DNP_use", "true"
]

print(f"\n🔍 Bắt đầu kiểm tra sức mạnh của model: {TEST_MODEL_PATH} ...\n", flush=True)

process_test = subprocess.Popen(
    cmd_test, cwd=REPO_DIR, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
)

for line in process_test.stdout:
    print(line, end="")
    sys.stdout.flush()

process_test.wait()

## 🔍 Bắt đầu kiểm tra sức mạnh của model: /kaggle/input/models/huynhvu7979/dsanet/pytorch/default/1/dsanet_model_xd.pth ...

### AUC1:  0.9539451211469483  AP1:  0.8699109897543252
### AUC2:  0.953945121014257  AP2: 0.8699109892289387
### mAP@0.1 =39.66%
### mAP@0.2 =32.20%
### mAP@0.3 =26.62%
### mAP@0.4 =20.93%
### mAP@0.5 =16.56%
### average MAP: 27.19


In [4]:
# ========================= TRAINING =========================
import json, os, subprocess, sys, time
from pathlib import Path
# Định nghĩa đường dẫn
ARTIFACTS = Path("/kaggle/working/rupa_artifacts")
REPO = Path("/kaggle/working/RUPA-DSA")
dataset_dir = ARTIFACTS / DATASET
dataset_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = dataset_dir / f"checkpoint_{DATASET}.pth"
model_path = dataset_dir / f"best_{DATASET}.pth"
train_csv = ARTIFACTS / f"{DATASET}_train.csv"
test_csv = ARTIFACTS / f"{DATASET}_test.csv"
cmd = [
    sys.executable, "-u", f"src/{DATASET}_train.py",
    "--train-list", str(train_csv),
    "--test-list", str(test_csv),
    "--model-path", str(model_path),
    "--checkpoint-path", str(checkpoint_path),
    
    "--max-epoch", str(MAX_EPOCHS),
    "--batch-size", str(XD_BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--seed", str(SEED),
    # ------------------ BẬT RUPA FINAL ------------------
    "--rupa-use", "true",
    
    # 1. Bật Otsu Thresholding
    "--adaptive_normal_selection", "true",
    
    # 2. Bật Warm Start (Kế thừa tri thức từ file DSANet gốc của tập XD)
    "--init-model-path", str(INIT_MODEL_PATH),
    
    # 3. Bật Safe Gate cho Routing
    "--routing-mode", "safe_gate",
    "--main-lr", "0.0",         # Đóng băng nhánh chính
    "--refiner-lr", "1e-5",     # Chỉ cho phép nhánh RUPA học tiếp
    # Các trọng số RUPA
    "--routing-det-weight", "0.5",  
    "--routing-rec-weight", "0.3",   
    "--routing-sem-weight", "0.2",   
    "--loss-residual-weight", "1.0",
    "--loss-reconstructed-normal-weight", "1.0",
    "--loss-dnp-normal-weight", "0.1",
    "--loss-consistency-weight", "1.0",
    "--loss-gather-weight", "1.0",
]
if RESUME:
    cmd.extend(["--use-checkpoint", "true"])
log_path = dataset_dir / f"train_{DATASET}_FINAL.log"
started = time.time()
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
print(f"🚀 Bắt đầu train XD-Violence bản FINAL (Otsu + WarmStart) với {MAX_EPOCHS} Epochs...\n", flush=True)
with log_path.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        cmd, cwd=REPO, env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        sys.stdout.flush()
        log_file.write(line)
        log_file.flush()
        
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"❌ Training thất bại với exit code {return_code}")
print(f"\n✅ Hoàn thành xuất sắc trong {(time.time() - started)/3600:.2f} giờ!")

🚀 Bắt đầu train XD-Violence bản FINAL (Otsu + WarmStart) với 10 Epochs...

Loading initial model weights from /kaggle/input/models/huynhvu7979/dsanet/pytorch/default/1/dsanet_model_xd.pth
epoch: 1 | step: 4800 | loss1: 0.1946 | loss2: 0.5212 | loss3: 0.0427 | loss4: 1.1258 | loss5: 0.6849 | consistency_loss: 0.0086 | g_loss: 0.2551 | dnp_normal_loss: 0.9790
epoch: 1 | step: 9600 | loss1: 0.2064 | loss2: 0.5315 | loss3: 0.0427 | loss4: 1.1347 | loss5: 0.6653 | consistency_loss: 0.0074 | g_loss: 0.2035 | dnp_normal_loss: 0.9716
epoch: 1 | step: 14400 | loss1: 0.2115 | loss2: 0.5394 | loss3: 0.0427 | loss4: 1.1418 | loss5: 0.6588 | consistency_loss: 0.0090 | g_loss: 0.1907 | dnp_normal_loss: 0.9653
epoch: 1 | step: 19200 | loss1: 0.2110 | loss2: 0.5444 | loss3: 0.0427 | loss4: 1.1429 | loss5: 0.6559 | consistency_loss: 0.0121 | g_loss: 0.1660 | dnp_normal_loss: 0.9489
epoch: 1 | step: 24000 | loss1: 0.2106 | loss2: 0.5466 | loss3: 0.0427 | loss4: 1.1373 | loss5: 0.6548 | consistency_loss: